# 06 · Organocatalysis: constraints across scaffolds

Transfer a conserved catalyst-core geometry between isothiourea backbones, then hold a reacting core in a
separate DFT transition state. Renders mark the constrained atoms and contacts.

In [ ]:
import rxembed as rx
import xyzrender
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")
CATALYSTS = {  # the isothiourea family; all share the amidine-thioether core, distinct backbones
    "tetramisole": "C1CSC2=NC(CN12)c1ccccc1",  # imidazo[2,1-b]thiazoline (no benzo fusion)
    "BTM": "C1CN2C(=NC1c1ccccc1)Sc1ccccc12",  # benzotetramisole: the thiazole is benzo-fused
    "HyperBTM": "CC(C)[C@]1CN2C(=N[C@]1c1ccccc1)Sc1ccccc12",  # BTM + an i-Pr stereocentre
}
CORE = "[#7][#6]([#16])=[#7]"  # the conserved amidine-thioether reacting core (N-C(S)=N)

## The catalyst family

Each backbone embeds and conformer-searches like anything else; the highlighted amidine–thioether core does the chemistry and is conserved across all of them.

In [ ]:
for name, smi in CATALYSTS.items():
    ens = rx.embed(smi, stereo="free").mc(preset="rapid").minimize().prune()  # free: undefined ring
    for report in ens.check().values():
        report.assert_ok()
    print(f"  {name:12s}: {len(ens):2d} conformers ({Chem.MolFromSmiles(smi).GetNumAtoms()} heavy atoms)")
hb = rx.embed(CATALYSTS["HyperBTM"]).mc(preset="rapid").minimize().prune()  # @ states the stereocentre
for report in hb.check().values():
    report.assert_ok()
highlight = [i + 1 for i in rx.match(hb.mol, CORE)]  # xyzrender uses 1-based indices
xyzrender.render(hb.mol, no_hy=True, highlight=highlight)

## A directional S···C=O contact

This neutral tetramisole–acetic anhydride pair is a pre-association model, not an acylated intermediate. `nci_candidates` finds the ring-S···carbonyl-O contact.

In [ ]:
catalyst = CATALYSTS["tetramisole"]
acyl_donor = "CC(=O)OC(C)=O"  # acetic anhydride: the acyl source
complex_smi = f"{catalyst}.{acyl_donor}"
cands = rx.nci_candidates(Chem.AddHs(Chem.MolFromSmiles(complex_smi)))
chb = next(k for k in cands if k.startswith("ChB"))  # the S...O=C chalcogen bond
print(f"candidate contact: {chb}  {cands[chb].distances}")
# stereo="free": tetramisole's ring stereocentre is undefined in the SMILES and irrelevant to the
# chalcogen grip; without it rx.embed returns an EnsembleSet of both enantiomers, not one Ensemble.
ens = rx.embed(complex_smi, contacts=cands[chb], n=8, stereo="free").mc().minimize().prune()
s, o = next(iter(cands[chb].distances))
ens.filter("geometry")
print(f"{len(ens)} conformers; S···O = {ens.measure((s, o))['mean']:.2f} Å")
xyzrender.render(
    ens.mol,
    no_hy=True,
    nci_bonds=[(s + 1, o + 1)],
    labels=[f"{s + 1} {o + 1} d"],
)  # the chalcogen contact (dotted)

## Transfer the core to another backbone

Embed tetramisole, then transfer its amidine-core geometry to the other backbones. `template=(reference, SMARTS)` matches the shared core on both graphs.

In [ ]:
template = rx.embed(CATALYSTS["tetramisole"], stereo="free").mc().minimize().prune()
swaps = {}
for name in ["BTM", "HyperBTM"]:
    sw = rx.embed(CATALYSTS[name], template=(template, CORE), stereo="free").mc().minimize().prune()
    for report in sw.check().values():
        report.assert_ok()
    swaps[name] = sw
    print(f"  tetramisole -> {name:9s}: {len(sw):2d} gate-clean conformers")
sw = swaps["HyperBTM"]
highlight = [i + 1 for i in rx.match(sw.mol, CORE)]
xyzrender.render(sw.mol, no_hy=True, highlight=highlight)

## Rank the transferred poses

Rank the searched poses with GFN-FF, then run g-xTB single points on the lowest few. Both require `xtb` on `$XTB_EXE`.

In [ ]:
ranked = swaps["HyperBTM"].score("gfnff")  # rank the templated poses on a real, NCI-aware energy
print(
    f"tetramisole->HyperBTM: {ranked.n} poses | GFN-FF ΔE {sorted(round(e, 1) for e in ranked.relative().values())[:5]} kcal/mol"
)
best = ranked.lowest(2).score("gxtb")  # g-xTB single point on the top 2
print(f"g-xTB on the top 2: ΔE {sorted(round(e, 2) for e in best.relative().values())} kcal/mol")

## A real DFT TS with a fixed core

For a TS `.xyz`, `fix=` holds the reacting atoms while the remaining torsions are searched. This chiral-phosphoric-acid TS has 103 atoms.

In [ ]:
cpa = "structures/cpa.xyz"
rc = [23, 35, 70, 74, 75]  # reacting-core atoms (from a graphrc trajectory in practice)
bonds = [(23, 35), (35, 75), (70, 74)]  # forming / breaking bonds held during embed
ens = rx.embed(cpa, fix=rc, n=2).mc().minimize().prune()
total = ens.n
ens.filter("geometry", frozen=rc, reference=cpa)
print(f"CPA organocatalysis TS (103 atoms): {len(ens)} conformers; fixed core checked against the XYZ")
print(f"geometry.check: {ens.n}/{total} clean")
xyzrender.render(ens.mol, no_hy=True, ts_bonds=[(i + 1, j + 1) for i, j in bonds])